In [14]:
from utils.data_preporcess import *
from visualization.visualize_schedule import visualize_results
#from config.logger_config import *
import time
from IPython.display import display, HTML
from scheduler_monolitic.legacy.gurobi_SCOS import *

In [15]:
## setup logger
logger = logging.getLogger(__name__)
logger.setLevel(logging.ERROR)

### prepare data for the optimization



In [16]:
# Load system data and historical nodal demand data
network, hourly_demand, config = data_loader('../PROPER/config/wp3_optim/IEEE24_scheduler.json')
aggregation_step = 'W'  # 'W', 'D', '12H', 'H,  etc.
# Define PM activities (planned outages)
comp_types = ['line', 'line', 'gen']
comp_ids = [0, 1, 2]
cost_per_days = [1000, 1000, 5000]  # m.u/day
expected_duration_days = [45, 75, 120]  # days/PM
priorities = [1, 2, 1]  # high = 3, medium = 2, low = 1
planned_outage_names = [f'{type}_{idx}' for idx, type, in zip(comp_ids, comp_types)]

2026-09-17 11:55:57,354::INFO::get_project_root::Current file path: C:\Users\roberto.rocchetta\Documents\GitHub\PROPER\utils\utils.py
2026-09-17 11:55:57,356::INFO::get_project_root::Checking directory: C:\Users\roberto.rocchetta\Documents\GitHub\PROPER\utils\utils.py
2026-09-17 11:55:57,358::INFO::get_project_root::Checking directory: C:\Users\roberto.rocchetta\Documents\GitHub\PROPER\utils
2026-09-17 11:55:57,362::INFO::get_project_root::Checking directory: C:\Users\roberto.rocchetta\Documents\GitHub\PROPER
2026-09-17 11:55:57,364::INFO::get_project_root::Found PROPER directory: C:\Users\roberto.rocchetta\Documents\GitHub\PROPER
2026-09-17 11:55:57,366::INFO::data_loader::Loading configuration from: C:\Users\roberto.rocchetta\Documents\GitHub\PROPER\config\wp3_optim\IEEE24_scheduler.json
2026-09-17 11:55:57,368::INFO::load_network::Loading network model
2026-09-17 11:55:57,696::INFO::correct_dtypes::These dtypes could not be corrected: {'trafo': ['tap_min', 'tap_max']}
2026-09-17 11:

In [17]:
### PREPARE DATA INPUT DICTIONARY
nodal_demand_aggregated = aggregate_hourly_demand(hourly_demand, aggregation_step=aggregation_step)
PM_cost_per_step, PM_duration_steps = aggregate_step_costs_and_durations(cost_per_days,expected_duration_days, aggregation_step=aggregation_step)

# example scheduled outage information
outages = {'indices': comp_ids, 
           'names': planned_outage_names, 
           'type': comp_types,
           'expected_duration_steps': PM_duration_steps,
           'cost_per_step': PM_cost_per_step,
           'priorities': priorities} 

num_branches = len(network.trafo) + len(network.line)
num_buses = len(network.bus)
branch_capacity = [175 if max_i_ka <= 1 else 500 for max_i_ka in network.line['max_i_ka']] + [400 for _ in network.trafo] 
n_minus1_names = [f'n1_{l}' for l in [f'line_{k}' for k in range(30)]]

DATA = {'max_tasks': 2, 
        'nodal_demand': nodal_demand_aggregated,
        'outages': outages, 
        'config': config, 
        'network': network,
        'num_buses': num_buses, 'num_branches': num_branches,
        'ref_buses': ['bus_12'], 'branch_capacity': branch_capacity,
        'n_minus1_names': n_minus1_names}

names = {
        'outages': DATA['outages']['names'],  # List of outage names
        'lines': [f'line_{ll}' for ll in range(DATA['num_branches'])],  # List of line names
        'contingencies': DATA.get('n_minus1_names', None),  # List of contingency names
        'buses': [f'bus_{b}' for b in range(DATA['num_buses'])],  # List of bus names
        'generators': [f'gen_{g}' for g in DATA['network'].gen.index.tolist()],  # List of generator names
        }

if names['contingencies'] is None:  # Generator indices
    names['contingencies'] = [f'n1_{l}' for l in names['lines']] + [f'n1_{g}' for g in names['generators']]
    
DATA['names'] = names

# Preprocess the data
T = [f'step_{t}' for t in range(len(DATA['nodal_demand']))]
DATA['p_max'] = {gn: (v + 100 if v > 0 else 200) for gn, v in zip(names['generators'], DATA['network'].gen['max_p_mw'])}
DATA['p_min'] = {gn: v * 0 for gn, v in zip(names['generators'], DATA['network'].gen['min_p_mw'])}  # todo fixme
DATA['f_lim'] = {ln: v for ln, v in zip(names['lines'], DATA['branch_capacity'])}
DATA['g2bus'] = [f'bus_{g}' for g in DATA['network'].gen['bus'].values.tolist()]

# Precompute generator to node mapping
DATA['gen_to_node'] = {b: names['generators'][idx] for idx, b in enumerate(DATA['g2bus'])}

# Pandapower/PYPOWER internal matrices
internal_ppc = DATA["network"]._ppc["internal"]
DATA['S'] = as_dense_array(internal_ppc["Cft"])
DATA['B_mat'] = as_dense_array(internal_ppc["Bf"])
DATA['B_lines'] = np.max(DATA['B_mat'], axis=1)

# Pre-fetch some values for efficiency
DATA['priority'] = {o_nam: DATA['outages']['priorities'][o] for o, o_nam in enumerate(names['outages'])}
DATA['durations'] = {o_nam: DATA['outages']['expected_duration_steps'][o] for o, o_nam in enumerate(names['outages'])}
DATA['step_cost_outage'] = {o_nam: DATA['outages']['cost_per_step'][o] for o, o_nam in enumerate(names['outages'])}

DATA['T'] = T  # value of loss load
  
DATA['VoLL'] = 1e6  # value of loss load
DATA['n_samples'] = 50
DATA['use_DC_PF'] = False

### start building the model

In [18]:
model_name = 'deterministic_optimizer' 
optimization_model = Model(f"Transmission_Outage_Scheduling_{model_name}") 

def add_variables(model: Model, data:dict, optim_type: str = 'deterministic'): 
    
    """ """ 
    O = data['names']['outages']
    G = data['names']['generators']
    L = data['names']['lines']
    B = data['names']['buses']
    C = data['names']['contingencies']
    T = data['T']
    
    #is_o     = model.addVars(O, vtype=GRB.BINARY, name="is_pm_scheduled")  # define variables
    xt       = model.addVars(T, O, vtype=GRB.BINARY, name="planned_outage_indicator")
    #xt_is_on = model.addVars(T, O, vtype=GRB.BINARY, name="planned_outage_indicator_active")
    sxt      = model.addVars(T, O, vtype=GRB.BINARY, name="start_outage_indicator")
    ext      = model.addVars(T, O, vtype=GRB.BINARY, name="end_outage_indicator")
    
    pgen = model.addVars(T, G, lb=0, name="power_generation")
    d_wc = model.addVars(T, B, lb=0, name="loss_of_load")
    f    = model.addVars(T, L, lb=-GRB.INFINITY, ub=GRB.INFINITY, name="flow_tl")
    
    f_c     = model.addVars(T, L, C, lb=-GRB.INFINITY, ub=GRB.INFINITY, name="flow_tl_contingency")
    d_wc_c  = model.addVars(T, B, C, lb=0, name="loss_of_load_contingency")
    pgen_c  = model.addVars(T, G, C, lb=0, name="power_generation_contingency")
    
    VARIABLES = {}
    #VARIABLES['is_o'] = is_o
    VARIABLES['xt'] = xt
    #VARIABLES['xt_is_on'] = xt_is_on
    VARIABLES['sxt'] = sxt
    VARIABLES['ext'] = ext
    VARIABLES['pgen'] = pgen
    VARIABLES['pgen_c'] = pgen_c
    VARIABLES['d_wc'] = d_wc
    VARIABLES['d_wc_c'] = d_wc_c
    VARIABLES['f'] = f
    VARIABLES['f_c'] = f_c 
    
    logger.info(f' \n'
                f'{blue_c}Variables Added:{reset_c}\n'
                f' - {blue_c}xt, sxt, ext{reset_c}: s Scheduled outage decisions, start-end indicators for the outage task\n'
                f' - {blue_c}pgen, pgen_c{reset_c}: Generated power planned and N-1 states\n'
                f' - {blue_c}d_cut, d_cut_c{reset_c}: loss of load planned and N-1 states\n'
                f' - {blue_c}f, f_c{reset_c}: Power flow in planned and N-1 states\n')
    
    if optim_type == 'deterministic':
        return model, VARIABLES
    
    elif optim_type == 'robust':
        # Add robust variables here if needed z = worst-case cost of ephi scenario formulation
        z = model.addVar(lb=-GRB.INFINITY, name="robust_obj")
        VARIABLES['z'] = z 
        return model, VARIABLES
        
    elif optim_type == 'risk_constrained':
       # Add variables for cvar computation
       S =  data['names']['demand_scenarios']
       d_wc_scenario = model.addVars(T, B, S, lb=0, name="Loss_of_bus_load_scenarios")
       excess = model.addVars(T, B, S, lb=0, name="Excess_over_VaR")
       VaR = model.addVars(T, B, lb = -float('inf'), name="Value_at_Risk")  # Unbounded VaR
       CVaR = model.addVars(T, B, lb=0, name="CVaR") 
       VARIABLES['d_wc_scenario'] = d_wc_scenario
       VARIABLES['excess'] = excess
       VARIABLES['VaR'] = VaR
       VARIABLES['CVaR'] = CVaR
       return model, VARIABLES

optimization_model, VARIABLES = add_variables(model=optimization_model, data=DATA, optim_type = 'deterministic')

### deterministic objective function

In [19]:
def objective_function(data, variables):
    # Unpack variables
    xt     = variables['xt'] 
    pgen   = variables['pgen']
    pgen_c = variables['pgen_c']
    d_wc   = variables['d_wc']
    d_wc_c = variables['d_wc_c'] 
    #is_o   = variables['is_o'] 

    T          = data['T']
    outages    = data['names']['outages']
    buses      = data['names']['buses']
    generators = data['names']['generators']
    contingencies = data['names']['contingencies']
    priority = data['priority']
    
    nt = len(T)

    # 1) PM priority score
    PM_priority = quicksum( (nt - t_idx) / nt * xt[t, o] * priority[o]    for t_idx, t in enumerate(T)  for o in outages )

    # scaling factors
    scale1 = nt * len(buses)
    scale2 = scale1 * len(contingencies)

    # 2) Loss of load penalty proportional to VoLL
    VOLL_cost = data['VoLL'] * quicksum(d_wc[t, n] for t in T for n in buses ) / scale1
    VOLL_cost_contingency = data['VoLL'] * quicksum(d_wc_c[t, n, c] for t in T for n in buses for c in contingencies) / scale2

    # 3) Operational costs
    OP_cost = quicksum( pgen[t, g] for t in T for g in generators ) / scale1
    OP_cost_contingency = quicksum( pgen_c[t, g, c] for t in T for g in generators for c in contingencies) / scale2

    # Optional: display LaTeX
    latex_expr = r"""
    <div style=\"color:orange;\">
    \[ \max_X \Bigl( \sum_o C_{PM,o}(X) - \omega_1 \sum_t\sum_b C_{VoLL,t,b}(X) \\
    - \omega_2 \sum_t\sum_b\sum_C C_{VoLL,t,b}(X|C)
    - \omega_3 \sum_t\sum_b C_{OP,b,t}(X)
    - \omega_4 \sum_t\sum_b\sum_C C_{OP,b,t}(X|C) \Bigr) \]
    </div>
    """
    display(HTML(latex_expr))
    # Collect objectives
    Objectives = {'PM_priority': PM_priority, 'VOLL_cost': VOLL_cost,
                  'VOLL_cost_contingency': VOLL_cost_contingency,
                  'OP_cost': OP_cost, 'OP_cost_contingency': OP_cost_contingency}
    
    Objectives['Total'] = (PM_priority - (VOLL_cost + VOLL_cost_contingency)  - (OP_cost + OP_cost_contingency))
    return Objectives

Objectives = objective_function(data=DATA, variables=VARIABLES)
optimization_model.setObjective(Objectives['Total'], GRB.MAXIMIZE)

###  ---  CONSTRAINTS:

In [20]:
def add_all_generator_constraints( model: Model, data: dict, variables: dict) -> Model:
    """
    Add generation bounds and contingency constraints in a single pass.
 
    Uses Gurobi's addConstrs for vectorized constraint creation.
    """
    
    logger.info('adding power production constraints, normal + N-1 failures')
    start = time.time()
    
    T           = data['T'] 
    gens        = data['names']['generators'] 
    conts       = data['names']['contingencies']
    outages_set  = set(names.get('outages', []))
    p_max       = data['p_max']
    p_min       = data['p_min']
    xt          = variables['xt']
    #is_o        = variables['is_o']  
    pgen        = variables['pgen']  
    pgen_c        = variables['pgen_c']  
    
    # Precompute factors for each (t,g) and normal generation bounds
    factors = {  (t, g): (xt[t, g] if g in outages_set else 0.0) for t in T for g in gens }
    model.addConstrs((pgen[t, g] <= p_max[g] * (1 - factors[t, g]) for t in T for g in gens),name="GenUp")
    model.addConstrs((pgen[t, g] >= p_min[g] * (1 - factors[t, g]) for t in T for g in gens), name="GenLow")

    # Contingency generation
    # Equality for failed g==c[3:]
    zero_constrs, bound_constrs_up, bound_constrs_low = [], [], [] 
    for t in T:
        for g in gens:
            for c in conts:
                if c[3:] == g:
                    zero_constrs.append((t, g, c))
                else:
                    bound_constrs_up.append((t, g, c))
                    bound_constrs_low.append((t, g, c))

    # Add zero-output constraints and Add bounds for non-failed contingencies
    model.addConstrs( (pgen_c[t, g, c] == 0 for t, g, c in zero_constrs),  name="GenNull" ) 
    model.addConstrs( (pgen_c[t, g, c] <= p_max[g] * (1 - factors[t, g])  for t, g, c in bound_constrs_up),  name="GenCUp"  )
    model.addConstrs( (pgen_c[t, g, c] >= p_min[g] * (1 - factors[t, g])  for t, g, c in bound_constrs_low), name="GenCLow" )
    logger.info(f"Done - build time:   {time.time() - start:.3f}s")
    return model


# noinspection PyTypeChecker
def add_planned_outages_constraints(model: Model, data: dict, variables: dict) -> Model:
    logger.info('adding constraints for scheduled outages: duration, number of tasks, continuity')
    start = time.time()
    out, maxT, dur, T = data['names']['outages'], data['max_tasks'], data['durations'], data['T']
    xt, st, ed = variables['xt'], variables['sxt'], variables['ext']
    # xt_is_on , is_o= variables['xt_is_on'] , variables['is_o']
    # 0) 
    # model.addConstrs((xt_is_on[t, o_i] == xt[t, o_i] * is_o[o_i] for o_i in out for t in T), name="MaxTasks")

    # 1) ≤ maxT simultaneous outages
    model.addConstrs((quicksum(xt[t, o_i]  for o_i in out) <= maxT for t in T), name="MaxTasks")
    # 2) exact total duration
    model.addConstrs((quicksum(xt[t, o_i]  for t in T) == dur[o_i]  for o_i in out), name="Duration")
    # 3) monotonicity & end-after-start
    model.addConstrs((st[T[i+1],o_i] >= st[T[i],o_i] for o_i in out for i in range(len(T)-1)), name="StartPM")
    model.addConstrs((ed[T[i+1],o_i] >= ed[T[i],o_i] for i in range(len(T)-1) for o_i in out), name="EndPM")
    model.addConstrs((ed[T[i+1],o_i] <= st[T[i],o_i] for i in range(len(T)-1) for o_i in out), name="EndAfterStart")
    # 4) link xt = st−ed
    model.addConstrs((st[t,o_i]-ed[t,o_i] == xt[t,o_i] for t in T for o_i in out), name="LinkPM")
    logger.info(f"Done - build time:   {time.time() - start:.3f}s")
    return model

def add_line_power_limit_constraints_fast(model: Model, data: dict, variables: dict) -> Model:
    logger.info('adding line flow constraints, normal + N-1 failures')
    start = time.time()
    lines, conts, outages, T = data['names']['lines'], data['names']['contingencies'], set(data['names']['outages']), data['T']
    f, f_c, xt, f_lim = variables['f'], variables['f_c'], variables['xt'],  data['f_lim']
    # xt_is_on , is_o= variables['xt_is_on'] , variables['is_o']
    
    
    factor = {(t,l): xt[t,l]  if l in outages else 0 for t in T for l in lines}
    cont_line = {c: next(l for l in lines if c.endswith(l)) for c in conts}
    model.addConstrs((f[t,l] <=  f_lim[l]*(1-factor[t,l]) for t in T for l in lines), name="FlowBase_UP")
    model.addConstrs((f[t,l] >= -f_lim[l]*(1-factor[t,l]) for t in T for l in lines), name="FlowBase_LOW")
    model.addConstrs((f_c[t,l,c]==0 if cont_line[c]==l else f_c[t,l,c] <=  f_lim[l]*(1-factor[t,l]) for t in T for l in lines for c in conts), name="FlowC_UP")
    model.addConstrs((f_c[t,l,c]==0 if cont_line[c]==l else f_c[t,l,c] >= -f_lim[l]*(1-factor[t,l]) for t in T for l in lines for c in conts), name="FlowC_LOW")
    logger.info(f"Done - build time:   {time.time() - start:.3f}s") 
    return model

def add_nodal_power_balance_constraints_fast(model: Model, data: dict, variables: dict) -> Model: 
    
    logger.info('adding node balance constraints, normal + N-1 failures')
    start = time.time() 
    names, S_T, demand, g2bus, T = data['names'], data['S'], data['nodal_demand'], data['g2bus'], data['T']
    pgen, pgen_c, f, f_c , d_wc, d_wc_c = (variables['pgen'], variables['pgen_c'],
                                           variables['f'], variables['f_c'], variables['d_wc'], variables['d_wc_c'])
    buses , contraints   = names['buses'] , names['contingencies']
    
    # 1) precompute full balance matrices
    base_bal = calculate_nodal_balance(T, f, names, S_T, pgen,   {b: names['generators'][i] for i,b in enumerate(g2bus)}, demand)
    cont_bal = {c: calculate_nodal_balance(T, f_c, names, S_T, pgen_c, {b: names['generators'][i] for i,b in enumerate(g2bus)}, demand, c=c) for c in contraints}

    # 2) build index ranges once
    T_idx = range(len(T))
    B_idx = range(len(buses))

    # 3) vectorized adds for base case
    model.addConstrs((base_bal[t, b] <= d_wc[T[t],  buses[b]] for t in T_idx for b in B_idx ), name="PB_up")
    model.addConstrs((base_bal[t, b] >= -d_wc[T[t], buses[b]] for t in T_idx for b in B_idx ), name="PB_low" )

    # 4) vectorized adds for each contingency
    for c in contraints:
        mb = cont_bal[c]
        model.addConstrs((mb[t, b] <= d_wc_c[T[t], buses[b], c] for t in T_idx for b in B_idx ),name=f"PB_{c}_up")
        model.addConstrs((mb[t, b] >= -d_wc_c[T[t], buses[b], c] for t in T_idx for b in B_idx ), name=f"PB_{c}_low")
    logger.info(f"Done - build time:   {time.time() - start:.3f}s") 
    return model
        

In [21]:
optimization_model = add_planned_outages_constraints(model=optimization_model, variables=VARIABLES, data=DATA) 
optimization_model = add_all_generator_constraints(model=optimization_model, variables=VARIABLES, data=DATA)
optimization_model = add_line_power_limit_constraints_fast(model=optimization_model, variables=VARIABLES, data=DATA)
optimization_model = add_nodal_power_balance_constraints_fast(model=optimization_model, variables=VARIABLES, data=DATA)

In [22]:
#@title SOLVE the Model
optimization_model.setParam('MIPGap', 0.01)             #  Acceptable optimality gap
optimization_model.setParam('Heuristics', 0.9)          # Emphasize heuristics
optimization_model.setParam('Cuts', 2)                  # Allow Gurobi to generate more cuts
optimization_model.setParam('Presolve', 2)              # Enable aggressive pre-solve
optimization_model.setParam('Threads', 10)              # Use 8 threads for parallel computation
optimization_model.setParam('TimeLimit', 3600)          # Set a one-hour time limit
optimization_model.setParam('MIPFocus', 1)              # Focus on finding feasible solutions
optimization_model.setParam('NodefileStart', 0.5)       # Start writing node files to disk at 0.5 GB
optimization_model.setParam('Method', 2)                # Use barrier method for root relaxation
optimization_model.update()

Set parameter MIPGap to value 0.01


2026-09-17 11:56:07,781::INFO::<module>::Set parameter MIPGap to value 0.01


Set parameter Heuristics to value 0.9


2026-09-17 11:56:07,790::INFO::<module>::Set parameter Heuristics to value 0.9


Set parameter Cuts to value 2


...

2026-09-17 11:56:07,824::INFO::<module>::Set parameter MIPFocus to value 1


Set parameter NodefileStart to value 0.5


2026-09-17 11:56:07,839::INFO::<module>::Set parameter NodefileStart to value 0.5


Set parameter Method to value 2


2026-09-17 11:56:07,845::INFO::<module>::Set parameter Method to value 2


In [23]:

optimization_model.optimize()

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11+.0 (26200.2))


2026-09-17 11:56:08,688::INFO::<module>::Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11+.0 (26200.2))


2026-09-17 11:56:08,698::INFO::<module>::


CPU model: 12th Gen Intel(R) Core(TM) i7-1265U, instruction set [SSE2|AVX|AVX2]


2026-09-17 11:56:08,704::INFO::<module>::CPU model: 12th Gen Intel(R) Core(TM) i7-1265U, instruction set [SSE2|AVX|AVX2]


...

2026-09-17 11:56:09,114::INFO::<module>::


Model is infeasible


2026-09-17 11:56:09,118::INFO::<module>::Model is infeasible


Best objective -, best bound -, gap -


2026-09-17 11:56:09,123::INFO::<module>::Best objective -, best bound -, gap -


In [24]:
# Check optimization status
def get_solution_dic(model):
    
    if model.status == GRB.OPTIMAL:
        logger.info(f"{green_c} Optimal solution found! :-) 🎉 :-) {reset_c}")
    elif model.status == GRB.TIME_LIMIT:
        logger.warning(f"{blue_c} Solution found with time limit! {reset_c}")
    else :
        logger.error(f"{red_c} Optimization ended with status: {model.status}{reset_c}")
    
     
    # Check optimization status
    SOLUTION = None
    if model.status in {GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.NODE_LIMIT, GRB.SUBOPTIMAL, GRB.USER_OBJ_LIMIT}:
        SOLUTION = {v.VarName: v.X for v in model.getVars()}  # Retrieve and save variable values 
   
    elif model.status == GRB.INFEASIBLE:
        logger.warning(f"{red_c} M is Infeasible! :-(:-(:-({reset_c}")
        logger.warning(f"{red_c} Run IIS to find conflicting constraints{reset_c}")
        model.computeIIS()
        logger.warning(f"{red_c} Writing IIS to a file for inspection {reset_c}")
        model.write("M.ilp")
        print("Conflicting constraints are:")
        for c in model.getConstrs():
            if c.IISConstr:
                print(c.ConstrName)
        return None
    
    elif model.status == GRB.UNBOUNDED:
        logger.warning("Model is unbounded.")  # Save unbounded M status to log or file
        with open("M_status.txt", "a") as f:
            f.write("Model is unbounded.\n")
        return None
 
    x_temp = []
    for o in names['outages']:
        x_temp.append([SOLUTION[f'planned_outage_indicator_active[{t},{o}]'] for t in T])
    X_OutageSchedule = pd.DataFrame(x_temp, index=names['outages'], columns=T)
    
    x_temp = []
    for l in names['lines']:
        x_temp.append([SOLUTION[f'flow_tl[{t},{l}]'] for t in T])
    FLOWS = pd.DataFrame(x_temp, index=names['lines'], columns=T)
    
    x_temp = []
    for c in names['contingencies']:
        x_temp.append(
            [sum([SOLUTION[f'loss_of_load_contingency[{t},{b},{c}]'] for b in names['buses']]) for t in
             T])
    WC_CURTAIL_CON = pd.DataFrame(x_temp, index=names['contingencies'], columns=T)
    WC_CURTAILED = pd.DataFrame(
        [sum([SOLUTION[f'loss_of_load[{t},{b}]'] for b in names['buses']]) for t in T], index=T).T
    
    x_temp = []
    for g in names['generators']:
        x_temp.append([SOLUTION[f'power_generation[{t},{g}]'] for t in T])
    GENERATION = pd.DataFrame(x_temp, index=names['generators'], columns=T)
    
    GEN_PLUS_CURTAILED = (GENERATION.sum().values + WC_CURTAILED.values)[0]
    
    results_variables_dictionary = {   
        "X_OutageSchedule": X_OutageSchedule,
        "PowerGenerated": GENERATION,
        "Line_Flows": FLOWS,
        "WC_CURTAIL": WC_CURTAILED,
        "WC_CURTAIL_CON": WC_CURTAIL_CON,
        "GEN_PLUS_CURTAILED": GEN_PLUS_CURTAILED,  
    }
    
    result_objective_function = model.getObjective().getValue()
    
    return results_variables_dictionary, result_objective_function

In [25]:
dictionary_variables, objective_function = get_solution_dic(optimization_model)
visualize_results(dictionary_variables, names)

2026-09-17 11:56:11,373::ERROR::get_solution_dic:: Optimization ended with status: 3


Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11+.0 (26200.2))


2026-09-17 11:56:11,387::INFO::get_solution_dic::Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11+.0 (26200.2))


2026-09-17 11:56:11,395::INFO::get_solution_dic::


CPU model: 12th Gen Intel(R) Core(TM) i7-1265U, instruction set [SSE2|AVX|AVX2]


...

2026-09-17 11:56:11,503::INFO::get_solution_dic::IIS computed: 1 constraints, 0 bounds


IIS runtime: 0.01 seconds (0.00 work units)


2026-09-17 11:56:11,503::INFO::get_solution_dic::IIS runtime: 0.01 seconds (0.00 work units)


Conflicting constraints are:
Duration[line_0]


TypeError: cannot unpack non-iterable NoneType object

In [ ]:
from scheduler_monolitic.gurobi_SCOS_cvar import demand_sampler
model_name = 'robust_optimizer'
optimization_model = Model(f"Transmission_Outage_Scheduling_{model_name}") 

optimization_model, VARIABLES = add_variables(optimization_model, data=DATA, optim_type = 'robust')

Objectives = objective_function(data=DATA, variables=VARIABLES)
optimization_model.setObjective(Objectives['Total'], GRB.MAXIMIZE)
 
### GENERATE SCENARIOS 
DATA['n_samples'] = 200
logger.info('adding node balance constraints for all demand scenarios (normal planned not contingencies)')
nodal_demand_samples = demand_sampler(DATA['nodal_demand'], n_samples=DATA['n_samples'], random_seed=42)
DATA['nodal_demand_samples'] = nodal_demand_samples

# e) bound z by scenario payoff
optimization_model.addConstr(VARIABLES['z'] <= Objectives['Total'],  name=f"robust_bound")

##CONSTRAITNS 
# perhaps we do not need additional variables but only need to add additional constraint for each demand scenario.... 
# Nodal power balance is soft because of d_wc and d_wc_contingency here so it should be fine....
# Line and generation limits are invariant w.r.t. the sceanrios so ok.....the 
logger.info(f'{blue_c} ADDING CONSTRAINTS: {reset_c}')
optimization_model = add_planned_outages_constraints(model=optimization_model, variables=VARIABLES, data=DATA)
optimization_model = add_all_generator_constraints(model=optimization_model, variables=VARIABLES, data=DATA)
optimization_model = add_line_power_limit_constraints_fast(model=optimization_model, variables=VARIABLES, data=DATA)
optimization_model = add_nodal_power_balance_constraints_fast(model=optimization_model, variables=VARIABLES, data=DATA) 

In [ ]:
def add_nodal_power_balance_scenarios(model: Model, data: dict, variables: dict) -> Model: 
    
    logger.info('adding node balance constraints, normal + N-1 failures')
    start = time.time()
    names, S_T, demand_samples, g2bus, T = data['names'], data['S'], data['nodal_demand_samples'], data['g2bus'], data['T']
    pgen, pgen_c, f, f_c , d_wc, d_wc_c, = variables['pgen'], variables['pgen_c'], variables['f'], variables['f_c'], variables['d_wc'], variables['d_wc_c']
    buses, contraints = names['buses'], names['contingencies'] 
    
    for s_idx, s in enumerate(names['demand_scenarios']):
        logger.info(f'adding node balance constraints, normal + N-1 failures....scenario {s}')
        demand = demand_samples[s_idx]

        # 1) precompute full balance matrices
        base_bal = calculate_nodal_balance(T, f,   names, S_T, pgen,   {b: names['generators'][i] for i,b in enumerate(g2bus)}, demand)
        cont_bal = {c: calculate_nodal_balance(T, f_c, names, S_T, pgen_c, {b: names['generators'][i] for i,b in enumerate(g2bus)}, demand, c=c) for c in contraints}
    
        # 2) build index ranges once
        T_idx = range(len(T))
        B_idx = range(len(buses))
    
        # 3) vectorized adds for base case
        model.addConstrs((base_bal[t, b] <= d_wc[T[t],  buses[b]] for t in T_idx for b in B_idx ), name="PB_up")
        model.addConstrs((base_bal[t, b] >= -d_wc[T[t], buses[b]] for t in T_idx for b in B_idx ), name="PB_low" )
    
        # 4) vectorized adds for each contingency
        for c in contraints:
            mb = cont_bal[c]
            model.addConstrs((mb[t, b] <= d_wc_c[T[t], buses[b], c] for t in T_idx for b in B_idx ),name=f"PB_{c}_up")
            model.addConstrs((mb[t, b] >= -d_wc_c[T[t], buses[b], c] for t in T_idx for b in B_idx ), name=f"PB_{c}_low")
    logger.info(f"Done - build time:   {time.time() - start:.3f}s") 
    return model

optimization_model = add_nodal_power_balance_scenarios(model=optimization_model, variables=VARIABLES, data=DATA)

In [ ]:
# ----  SOLVE the M
optimization_model.setParam('MIPGap', 0.1)  #  Acceptable optimality gap
optimization_model.setParam('Heuristics', 0.9)  # Emphasize heuristics
optimization_model.setParam('Cuts', 2)  # Allow Gurobi to generate more cuts
optimization_model.setParam('Presolve', 2)  # Enable aggressive pre-solve
optimization_model.setParam('Threads', 10)  # Use 8 threads for parallel computation
optimization_model.setParam('TimeLimit', 3600)  # Set a one-hour time limit
optimization_model.setParam('MIPFocus', 1)        # Focus on finding feasible solutions
optimization_model.setParam('NodefileStart', 0.5) # Start writing node files to disk at 0.5 GB
optimization_model.setParam('Method', 2)          # Use barrier method for root relaxation 
optimization_model.update()

optimization_model.optimize()

In [ ]:
dictionary_variables, objective_function = get_solution_dic(optimization_model)  # get solution dictionary with the optimized variables 
visualize_results(dictionary_variables, names)  # print some schedule_results and visuals

In [ ]:

model_name = 'risk_constrained_optimizer'
optimization_model = Model(f"Transmission_Outage_Scheduling_{model_name}") 

optimization_model, VARIABLES = add_variables(optimization_model, optim_type = 'risk_constrained')
Objectives = objective_function(data=DATA, variables=VARIABLES)
optimization_model.setObjective(Objectives['Total'], GRB.MAXIMIZE)
 
##CONSTRAITNS 
# perhaps we do not need additional variables but only need to add additional constraint for each demand scenario.... 
# Nodal power balance is soft because of d_wc and d_wc_contingency here so it should be fine....
# Line and generation limits are invariant w.r.t. the sceanrios so ok.....the 
logger.info(f'{blue_c} ADDING CONSTRAINTS: {reset_c}')
optimization_model = add_planned_outages_constraints(model=optimization_model, variables=VARIABLES, data=DATA)
optimization_model = add_all_generator_constraints(model=optimization_model, variables=VARIABLES, data=DATA)
optimization_model = add_line_power_limit_constraints_fast(model=optimization_model, variables=VARIABLES, data=DATA)
optimization_model = add_nodal_power_balance_constraints_fast(model=optimization_model, variables=VARIABLES, data=DATA)

# these are the additioan lvariables for the CVaR Formulation 
d_wc_scenario =  VARIABLES['d_wc_scenario'] 
excess =  VARIABLES['excess'] 
VaR =      VARIABLES['VaR']
CVaR =    VARIABLES['CVaR'] 


# define CVaR Constraint CVaR_{1-alpha}(x) = 1/alpha * E[excess] + VaR <= max 
DATA['alpha'] = 0.95
DATA['limits']['max_CVaR'] = 200
alpha = DATA['alpha']        # e.g. 0.95
S = DATA['names']['demand_scenarios']
T = DATA['time_index']
B = DATA['bus_index']
max_cvar = DATA['limits']['max_CVaR']  # shape (T,B)

# 1) excess >= d_wc - VaR, and excess >= 0 (if you haven’t already enforced nonnegativity)
for t in T:
    for b in B:
        for s in S:
            optimization_model.addConstr(
                excess[t,b,s] >= d_wc_scenario[t,b,s] - VaR[t,b],
                name=f"excess_def_t{t}_b{b}_s{s}"
            )
            # nonnegativity of excess is already via lb=0 on the var

# 2) CVaR definition: VaR + (1/α) * E_s[excess]
inv_alpha = 1.0/alpha
for t in T:
    for b in B:
        optimization_model.addConstr(
            CVaR[t,b] == VaR[t,b] 
                        + inv_alpha * (1.0/len(S)) 
                          * sum(excess[t,b,s] for s in S),
            name=f"cvar_def_t{t}_b{b}"
        )

# 3) Optional: enforce CVaR ≤ some limit
for t in T:
    for b in B:
        optimization_model.addConstr(
            CVaR[t,b] <= max_cvar[t,b],
            name=f"cvar_ub_t{t}_b{b}"
        )